# 02 — Phân tích thăm dò dữ liệu (EDA) và kiểm định thống kê

**Nhóm 10 — IE313.F32.LT.CNTT.** Nguồn dữ liệu: Inside Airbnb Bangkok (`data/processed/bangkok_cleaned.csv`). Hạn nộp: 18/10/2026.

Ánh xạ bài giảng:
- **Bài 05 (EDA)**: Thống kê mô tả (Mean, Median, Std, IQR, Skewness, Kurtosis), Kiểm định tương quan Pearson ($r$, $p$-value), Bảng chéo 2D (Groupby, Pivot table), Đánh giá điều kiện và kiểm định One-way ANOVA ($F$, $p$-value).
- **Bài 02 (Tiền xử lý)**: Minh họa Feature Scaling (Simple feature scaling, Min-Max, Z-score), Binning (Equal-width vs Quantile), và One-Hot Encoding (Dummy variables) trên bản sao EDA cách ly hoàn toàn, không ghi đè baseline.
- **Ranh giới học thuật**: Tương quan không phải nhân quả. Các kiểm định thống kê mang tính khám phá (exploratory), lưu ý rủi ro gom cụm chủ nhà (host clustering).

Levene và qcut được tách nhãn **[MỞ RỘNG NGOÀI BÀI HỌC]**, xem `docs/EDA_METHODS.md` về nguồn, mục đích và giới hạn. EDA moments dùng bias=False; khác audit sơ bộ bias=True.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(
    (
        p
        for p in candidates
        if (p / "data/source_manifest.json").is_file() and (p / "src/eda_stats.py").is_file()
    ),
    None,
)
if PROJECT is None:
    raise RuntimeError("Chạy từ gốc đồ án hoặc thư mục notebooks.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.data_loader import compute_sha256  # noqa: E402
from src.eda_stats import (  # noqa: E402
    EXPECTED_BASELINE_SHA256,
    compute_anova_test,
    compute_neighbourhood_pivot,
    compute_pairwise_correlations,
    compute_price_distribution_stats,
    compute_room_type_stats,
    demonstrate_lesson02_wrangling,
    run_full_eda,
)

baseline_csv = PROJECT / "data/processed/bangkok_cleaned.csv"
actual_hash = compute_sha256(baseline_csv)
assert actual_hash == EXPECTED_BASELINE_SHA256, f"Hash baseline không khớp: {actual_hash}"
df = pd.read_csv(baseline_csv)

print(
    f"Xác thực baseline thành công: {df.shape[0]:,} dòng x {df.shape[1]} cột. SHA-256: {actual_hash}"
)

## 1. Khám phá biến mục tiêu trung tâm: Giá thuê theo đêm (RQ1)

Theo Bài 05 tr. 4–20, tính toán đầy đủ các đại lượng thống kê mô tả: độ tập trung (Mean, Median, Mode), độ phân tán (Min, Max, Range, IQR, Variance, Std, CV), và hình dạng phân phối (Skewness, Kurtosis).

In [ ]:
st_price = compute_price_distribution_stats(df["price"])
df_rq1 = pd.DataFrame([st_price])
display(
    df_rq1[
        [
            "n",
            "min",
            "q1",
            "median",
            "mean",
            "q3",
            "max",
            "iqr",
            "std",
            "cv_pct",
            "skewness",
            "kurtosis",
            "outliers_count",
            "outliers_pct",
        ]
    ]
)

print(f"- Giá trung vị (Median): {st_price['median']:,.2f} THB/đêm")
print(
    f"- Giá trung bình (Mean): {st_price['mean']:,.2f} THB/đêm (cao gấp {st_price['mean'] / st_price['median']:.2f} lần Median; skewness = +{st_price['skewness']:.2f})"
)
print(f"- Khoảng biến thiên tứ phân vị (IQR): {st_price['iqr']:,.2f} THB/đêm")
print(
    f"- Số ngoại lai theo 1.5xIQR (> {st_price['upper_bound']:,.2f} THB): {st_price['outliers_count']:,} căn ({st_price['outliers_pct']}%)"
)

## 2. Giá theo loại phòng và ANOVA thăm dò (RQ2)

Mô tả nhóm bằng n, trung vị, Q1/Q3 và độ lệch. Skew không phải kiểm định chuẩn; không dùng p≥0,05 để chứng minh phương sai bằng nhau, không dùng ID duy nhất để chứng minh độc lập.

**[MỞ RỘNG NGOÀI BÀI HỌC]** Levene center=median chỉ chẩn đoán phương sai, không thay thế phương pháp chính trong Bài 05. Nguồn: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.levene.html ; host clustering giới hạn p-value. Xem `docs/EDA_METHODS.md`.

ANOVA tổng thể minh họa F/p thăm dò; không thực hiện so sánh cặp khi điều kiện suy luận chưa xác nhận.

In [ ]:
rq2_stats = compute_room_type_stats(df)
df_room = pd.DataFrame.from_dict(rq2_stats["by_room_type"], orient="index")
display(df_room[["n", "group_n_pct", "q1", "median", "q3", "mean", "std", "skewness"]])
anova_res = compute_anova_test(df)
display(pd.DataFrame([anova_res["omnibus_test"]]))
print(anova_res["condition_checks"]["scientific_recommendation"])
print("Levene (mở rộng):", anova_res["condition_checks"]["homoscedasticity_evaluation"])
print("Host clustering trong baseline:", anova_res["condition_checks"]["independence_evaluation"])
print("So sánh cặp:", anova_res["pairwise_status"])

## 3. Mối liên hệ tuyến tính giữa quy mô chỗ ở và giá thuê (RQ3)

Tính toán hệ số tương quan tuyến tính Pearson $r$ và giá trị $p$-value giữa 4 biến quy mô (`accommodates`, `bedrooms`, `bathrooms`, `beds`) và `price` theo Bài 05 tr. 24–27.
Báo cáo rõ số mẫu hợp lệ theo từng cặp (pairwise complete cases) và số dòng bị khuyết ở biến tương ứng.

In [ ]:
corr_df = compute_pairwise_correlations(df)
display(corr_df)

print("NHẬN XÉT PHƯƠNG PHÁP LUẬN TƯƠNG QUAN (RQ3):")
for _, r in corr_df.iterrows():
    p_disp = "< 0.001" if r["p_value"] < 0.001 else f"= {r['p_value']:.4e}"
    print(
        f"- {r['feature']:15}: r = {r['pearson_r']:+.4f} (p {p_disp}, n hợp lệ = {r['n_valid']:,}, khuyết = {r['n_missing']:,}) -> {r['strength']}"
    )
print(
    "- p nhỏ không biểu thị tương quan mạnh; host clustering giới hạn suy luận. Chưa xác định nguyên nhân r thấp hay tác động của ngoại lệ."
)

## 4. Phân hóa địa lý theo quận hành chính và loại phòng (RQ4)

Xây dựng bảng chéo 2D (Pivot table) biểu diễn giá trung vị, số lượng mẫu $n$, và khoảng tứ phân vị IQR cho 50 quận hành chính Bangkok $\times$ 4 loại phòng theo Bài 05 tr. 28–36.
Cảnh báo các ô có số lượng mẫu rất nhỏ ($n < 5$) để tránh ngụy biện thống kê.

In [ ]:
p_med, p_cnt, p_iqr, p_meta = compute_neighbourhood_pivot(df)
top_districts = df["neighbourhood_cleansed"].value_counts().head(15).index
print("Trung vị THB/đêm và số mẫu theo quận × loại phòng:")
display(p_med.loc[top_districts])
display(p_cnt.loc[top_districts])
print({key: p_meta[key] for key in ["total_pivot_cells", "empty_cells_count", "low_n_cells_count"]})
room = "Entire home/apt"
within_room = pd.DataFrame({"median": p_med[room], "n": p_cnt[room]})
within_room = within_room[within_room.n >= 20].sort_values("median")
print("So sánh trong cùng Entire home/apt; n≥20 là quy tắc hiển thị:")
display(pd.concat([within_room.head(5), within_room.tail(5)]))

## 5. Minh họa kỹ thuật tiền xử lý Bài 02 (Scaling, Binning, Dummy)

Thực hiện minh họa các kỹ thuật của Bài 02 (TH01, Bài 02) trên một bản sao cách ly (`eda_sample`), không làm biến đổi tệp baseline và không rò rỉ dữ liệu vào mô hình hồi quy sau này:
1. Feature Scaling trên `accommodates`: Simple Feature Scaling, Min-Max Normalization, Z-score Standardization.
2. Binning trên `price`: So sánh phân khoảng đều (`pd.cut`) vs phân khoảng theo phân vị (`pd.qcut`).
3. Mã hóa biến giả (One-Hot Encoding) trên `room_type` qua `pd.get_dummies`.

**[MỞ RỘNG NGOÀI BÀI HỌC]** qcut minh họa phân khoảng theo phân vị; nguồn https://pandas.pydata.org/docs/reference/api/pandas.qcut.html . Nhóm trùng giá có thể không chia đủ khoảng. Không khẳng định ưu thế chung; xem `EDA_METHODS.md`.

In [ ]:
l02_demo = demonstrate_lesson02_wrangling(df)
display(pd.DataFrame(l02_demo["scaling_summary"]))
print("So sánh 2 phương pháp Binning:")
print("- Equal-width binning (pd.cut):", l02_demo["binning_summary"]["equal_width_counts"])
print("- Quantile-based binning (pd.qcut):", l02_demo["binning_summary"]["quantile_counts"])
print(f"- Bài học sư phạm: {l02_demo['binning_summary']['pedagogical_lesson']}")

print("Mẫu dữ liệu One-Hot Encoding:")
display(pd.DataFrame(l02_demo["dummy_sample"]))

## 6. Xuất bản toàn bộ tệp thống kê máy đọc & Kiểm tra bàn giao

Thực thi hàm `run_full_eda()` để xuất bản toàn bộ 12 artifact thống kê sang `reports/stats/` có gắn metadata và hash xác thực.

In [ ]:
meta_eda = run_full_eda(
    baseline_csv, PROJECT / "reports/stats", expected_sha256=EXPECTED_BASELINE_SHA256
)
assert (PROJECT / "reports/stats/rq1_price_distribution.json").is_file()
assert (PROJECT / "reports/stats/rq2_room_type_stats.csv").is_file()
assert (PROJECT / "reports/stats/rq3_size_correlation.csv").is_file()
assert (PROJECT / "reports/stats/rq4_neighbourhood_pivot_median.csv").is_file()
print("Toàn bộ artifact thống kê máy đọc đã sẵn sàng tại reports/stats/.")
print("Bước tiếp theo: notebook 03 — trực quan hóa biểu đồ 300 DPI theo 5 bước Bài 04.")